# FM study, post hoc check G1: pad-to-square re-extraction (MIRAGE and Inception-v3)

One GPU session, about 1 hour (estimate, not measured). It re-extracts features for **MIRAGE and
Inception-v3 only**, with every image first padded with black to a square around its centre and then
given to the model's own locked input transform. Everything else is the locked study: the same images,
the same weights (checked against their pinned SHA256), the same code for every hashed file. It reads
**no labels and produces no scores**; scoring runs on the laptop (`fm_posthoc.py g1`).

This is a labelled post hoc sensitivity check (protocols.md section 8, G1), outside amendment A2 A-11.
It never replaces a locked result. Guide: `docs/research/FM_FIRST_RUN_GUIDE.md`, Part G1.

**Runtime:** GPU **A100, L4 or T4** (not Blackwell). No Hugging Face token is needed: MIRAGE and the
Inception weights are public downloads. Your My Drive needs the shortcut to Lam's `octfsl` folder.

| Step | What it does | What proves it worked |
|---|---|---|
| 1 | Checks the first cell and the GPU | `RUN_BY` filled in; a supported GPU |
| 2 | Clones the exact code; installs the locked Python 3.13.16 and the study environment (as the evaluate notebook) | Commit matches; `pip check` passes; PyTorch sees the GPU |
| 3 | Unpacks the images from `octfsl/raw`, then disconnects Drive | All 84,630 manifest images exist |
| 4 | Restores the locked input lock and external query lock from `octfsl/fm_v1/final/` | Both SHA256 match; lock restored |
| 5 | Downloads MIRAGE and the Inception weights | Every file matches its pinned SHA256 |
| 6 | Downloads OCTID and OCTDL (as the locked notebook) | Every file matches its published MD5 |
| 7 | Keeps Drive and credentials away from the model code | Drive is off and no login is cached |
| 8 | Pads and extracts, one process per model | `exit code 0` twice |
| 9 | Saves both feature folders to `octfsl/fm_v1/final/<attempt>/g1/`, then disconnects Drive | Every Drive copy's SHA256 matches |

**Fill in `RUN_BY` in the next cell, then Runtime → Run all.** Keep the runtime alive until Step 9 ends.

In [ ]:
# Fill in this cell before Run all (Step 1 stops until it is complete)
RUN_BY = ""  # your name
ATTEMPT = "final-20261113T170804Z"  # the locked attempt; do not change
LOCK_SHA256 = "239838279b77bb37211398a016c1c766273a35f5a06649b31f5cd97f562c001c"
EXTERNAL_SHA256 = "3414889b53ee10c526c4d3dfb28b7477676f5bf449c2729008902f93447f45d0"
# Used by the reused external-download cell only; copied verbatim from the locked notebook's first cell.
EXTERNAL_DECISION = ('Lam, 2026-11-11: "Approve" (download OCTID and OCTDL; cohorts fixed by amendment A1, '
                     "12 November 2026).")

In [ ]:
# Settings and helpers
import csv, datetime, hashlib, json, os, pathlib, shlex, shutil, subprocess, sys, time, zipfile

COMMIT = "39c3d73700a712584915f12fd542aec6501a086c"  # the locked code; change only on purpose
REPO = "https://github.com/Malnoad/Few-shot-rare-diseases"
ROOT, ENV = "/content/Few-shot", "/content/fm-env"
PY = f"{ENV}/bin/python"
PREPARED, ASSETS = "/content/prepared", "/content/assets"  # PREPARED becomes the input-lock folder at Step 12
WORK = "/content/fm_v1"  # specs, ledger, pilots, logs, reports, external query lock and features
LEDGER, REPORTS = f"{WORK}/download_ledger.csv", f"{WORK}/reports"
FEATURES, EXTERNAL_LOCK, META = f"{WORK}/features", f"{WORK}/external", f"{WORK}/external_meta"
DOWNLOADS, EXPORTS = "/content/external_downloads", "/content/fm_exports"  # big downloads; ZIPs made for Drive
LOCK_ZIP = f"{EXPORTS}/input-lock.zip"
RAW = "/content/drive/MyDrive/octfsl/raw"  # Lam's folder, through the runner's octfsl shortcut (guide, Part 1)
OUT = "/content/drive/MyDrive/octfsl/fm_v1"  # Lam's output folder, through the same shortcut
IMAGES = 84630
ARMS = ("RETFound", "DINOv2", "MIRAGE", "Inception")  # three FMs and the reference (A3 S-4: no fallback)

# Produced on the laptop from the same code and manifest (9 Nov).
EXPECTED_PREPARED = {
    "groups.csv": "6991d8229149aadf4af5ee53f1e2eb3991805d3bf56ebb5b367337a30224c5d0",
    "nine_way_roster.csv": "72f90c00b98810af46d257fe53d047d93f5312552ebaa478aef72d58da81b586",
    "episodes_development.csv": "cd060bddb434b5de476f4fd3550b8b21481ebaf3eb597298f727517968544695",
    "episodes_reporting.csv": "f9e1e3c6a4f2aacd405171837d472190271da2b03dbce79b619a03900b202b8f",
    "timing_inventory.csv": "a6be7a5ec3ba1bbe220674faaef8e51105bc7e4da59e9672d20e4e4562be97ed",
    "extraction_inventory.csv": "5ca665e4828b2d45c7d90cc6363708d9548793de7019d417e57222ee2b76b44d",
}


def sh(command):
    """Run a shell command, streaming its output; any failure stops the notebook."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"FAILED: {command}")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def report(name, lines):
    """Print a report and keep a copy in REPORTS, so the save step can put it on Drive."""
    text = "\n".join(lines)
    print(text)
    path = pathlib.Path(REPORTS, name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text + "\n", encoding="utf-8")


def save_to_drive(folder, paths, replace=False):
    """Copy files into octfsl/fm_v1/<folder>, verify every copy, then disconnect Drive.
    A file already there with the same bytes is kept; different bytes are never overwritten,
    except a receipt saved with replace=True."""
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    try:
        target = pathlib.Path(OUT, folder)
        target.mkdir(parents=True, exist_ok=True)
        for path in map(pathlib.Path, paths):
            digest, copy = sha256(path), target / path.name
            if copy.exists() and not replace:
                assert sha256(copy) == digest, f"{copy} already exists with different bytes; nothing was overwritten"
            else:  # copy under a temporary name first, so an interrupted copy never blocks a retry
                pending = target / (path.name + ".part")
                shutil.copyfile(path, pending)
                assert sha256(pending) == digest, (f"The Drive copy of {path.name} differs: run this step again; if it fails "
                                                   f"again, download {path} from the Files panel and send it to Lam.")
                os.replace(pending, copy)
            print(f"saved {path.name}  {digest}")
    finally:
        drive.flush_and_unmount()

In [ ]:
# Step 1. Check the first cell and the GPU
assert RUN_BY.strip() and "[" not in RUN_BY and "<" not in RUN_BY, "Stopped on purpose: write your name in RUN_BY, run that cell, then Run all."
G1_ARMS = ("MIRAGE", "Inception")  # G1 re-extracts these two arms only (ARMS stays the locked four)
G1 = "/content/g1"
query = "nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader"
gpu = subprocess.run(query, shell=True, capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU: Runtime > Change runtime type > GPU (A100, L4 or T4)"
print(gpu.stdout.strip())
assert float(gpu.stdout.split(",")[1]) < 12, "Blackwell GPU: switch to A100, L4 or T4"

In [ ]:
# Step 2. Get the exact code and build the same Python environment as at the lock
# Colab's own Python can change between sessions (3.13.16 at the lock, 3.13.18 on 14 Nov). The lock fixed
# Python 3.13.16 (input_lock.json, evaluation_environment), so this step installs that exact build,
# checked by SHA256 (python-build-standalone release 20261020), and builds the environment with it.
PYTHON_VERSION = "3.13.16"
PYTHON_URL = ("https://github.com/astral-sh/python-build-standalone/releases/download/20261020/"
              "cpython-3.13.16%2B20261020-x86_64-unknown-linux-gnu-install_only.tar.gz")
PYTHON_SHA256 = "0ad293ecda8ea93a28e4a6b5e9cc20ae74be15aafed75be04418b8f82bd0d0da"
BASE = f"/content/python-{PYTHON_VERSION}"
if not os.path.isdir(ROOT):
    sh(f"git clone -q {REPO} {ROOT}")
sh(f"git -C {ROOT} checkout -q {COMMIT}")
head = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, head
print("code:", head)


def python_version(python):
    return subprocess.check_output([python, "-c", "import platform; print(platform.python_version())"], text=True).strip()


if not os.path.exists(f"{BASE}/bin/python3"):
    tarball = f"{BASE}.tar.gz"
    sh(f"curl -sSfL -o {tarball} {PYTHON_URL}")
    assert sha256(tarball) == PYTHON_SHA256, "The Python download differs from the pinned build: run this step again; if it fails again, tell Lam."
    shutil.rmtree(f"{BASE}.part", ignore_errors=True)
    os.makedirs(f"{BASE}.part")
    sh(f"tar -xzf {tarball} -C {BASE}.part")
    os.replace(f"{BASE}.part/python", BASE)  # complete or absent, so a rerun never sees half a Python
    shutil.rmtree(f"{BASE}.part", ignore_errors=True)
assert python_version(f"{BASE}/bin/python3") == PYTHON_VERSION, "The installed Python is not the locked version: tell Lam."
if os.path.exists(PY) and python_version(PY) != PYTHON_VERSION:
    shutil.rmtree(ENV)  # an environment built earlier with Colab's own Python
if not os.path.exists(PY):
    sh(f"{BASE}/bin/python3 -m venv --without-pip {ENV}")
sh(f"python3 -m pip --python {PY} install -q -r {ROOT}/requirements-fm.txt")
sh(f"python3 -m pip --python {PY} check")
sh(f"""{PY} -c 'import sys, numpy, scipy; print("python", sys.version.split()[0], "numpy", numpy.__version__, "scipy", scipy.__version__)'""")

In [ ]:
# Step 2b. PyTorch in the study environment sees the GPU (MIRAGE); Inception uses Colab's own TensorFlow, as at the lock
sh(f"""{PY} -c 'import torch; assert torch.cuda.is_available(), "PyTorch cannot see the GPU"; print("torch", torch.__version__, "cuda", torch.version.cuda)'""")
sh("""python3 -c 'import sys, tensorflow as tf; print("Colab python", sys.version.split()[0], "tensorflow", tf.__version__, "(the lock had 3.13.16 and 2.21.0; differences are recorded and measured by the control images)")'""")

In [ ]:
# Step 3. Check Drive, unpack the images, then disconnect Drive
from google.colab import drive


def drive_ready():
    """Check both folders now, so a Drive problem cannot surface only at Step 11."""
    for path in (RAW, OUT):
        assert os.path.isdir(path), f"{path} not found: add the octfsl shortcut to your My Drive (guide, Part 1)"
    probe = pathlib.Path(OUT, f".write-test-{time.time_ns()}")
    probe.write_text("ok", encoding="utf-8")
    assert probe.read_text(encoding="utf-8") == "ok", "octfsl/fm_v1 is not writable: ask Lam to check Editor access"
    probe.unlink()
    print("Drive ok: octfsl/raw is readable and octfsl/fm_v1 is writable")


def extract(zip_path, anchor):
    """Copy every file below the folder named `anchor` into data/raw, skipping macOS junk."""
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            parts = [p for p in info.filename.replace("\\", "/").split("/") if p]
            if not parts or info.filename.endswith(("/", "\\")) or "__MACOSX" in parts or parts[-1].startswith("._"):
                continue
            if anchor not in parts[:-1]:
                continue
            target = pathlib.Path(ROOT, "data/raw", *parts[parts.index(anchor):])
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            count += 1
    print(f"{pathlib.Path(zip_path).name}: {count} files")


with open(f"{ROOT}/data/manifest.csv", newline="", encoding="utf-8") as stream:
    MANIFEST = [row["path"].replace("\\", "/") for row in csv.DictReader(stream)]


def absent():
    return [p for p in MANIFEST if not os.path.isfile(f"{ROOT}/{p}")]


assert len(MANIFEST) == IMAGES, len(MANIFEST)
if absent() or not globals().get("DRIVE_OK"):
    drive.mount("/content/drive", force_remount=True)
    drive_ready()
    if absent():
        extract(f"{RAW}/oct2017.zip", "OCT2017")
        extract(f"{RAW}/rare.zip", "rare")
    drive.flush_and_unmount()
    DRIVE_OK = True
missing_images = absent()
assert not missing_images, f"{len(missing_images)} manifest images missing; first: {missing_images[0]}"
print(len(MANIFEST), "manifest images present")

In [ ]:
# Step 4. Restore the locked input lock and external query lock from Drive (read-only copies)
from google.colab import drive
os.makedirs(f"{G1}/archives", exist_ok=True)
drive.mount("/content/drive", force_remount=True)
try:
    for name in ("input-lock.zip", "external-lock.zip"):
        shutil.copyfile(f"{OUT}/final/{ATTEMPT}/{name}", f"{G1}/archives/{name}")
finally:
    drive.flush_and_unmount()
assert sha256(f"{G1}/archives/input-lock.zip") == LOCK_SHA256, "input-lock.zip differs from the locked SHA256: stop and tell Lam."
assert sha256(f"{G1}/archives/external-lock.zip") == EXTERNAL_SHA256, "external-lock.zip differs from the locked SHA256: stop and tell Lam."
sh(f"{PY} {ROOT}/fm_prepare.py restore --root {ROOT} --out {G1}/lock --archive {G1}/archives/input-lock.zip --sha256 {LOCK_SHA256}")
with zipfile.ZipFile(f"{G1}/archives/external-lock.zip") as archive:
    archive.extractall(f"{G1}/external")
print("Locks restored:", LOCK_SHA256[:12], EXTERNAL_SHA256[:12])

In [ ]:
# Step 5a. Define the checkpoint downloads (revisions and digests come from fm_extract.py)
import requests

pins = json.loads(subprocess.check_output([PY, "-c", (
    "import json, fm_extract as x; print(json.dumps(dict(repositories=x.REPOSITORIES, revisions=x.REVISIONS, "
    "sources=x.SOURCE_REVISIONS, digests=x.EXPECTED_ASSET_SHA256)))")], cwd=ROOT, text=True))


def hf(arm, name):
    return f"https://huggingface.co/{pins['repositories'][arm]}/resolve/{pins['revisions'][arm]}/{name}"


def github(repo, arm, path):
    return f"https://raw.githubusercontent.com/{repo}/{pins['sources'][arm]}/{path}"


KERAS = "inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
FILES = [  # arm, file name, source, needs the Hugging Face token
    ("RETFound", "RETFound_mae_natureOCT.pth", hf("RETFound", "RETFound_mae_natureOCT.pth"), True),
    ("RETFound", "models_vit.py", github("rmaphoh/RETFound_MAE", "RETFound", "models_vit.py"), False),
    ("DINOv2", "model.safetensors", hf("DINOv2", "model.safetensors"), False),
    ("DINOv2", "config.json", hf("DINOv2", "config.json"), False),
    ("DINOv2", "preprocessor_config.json", hf("DINOv2", "preprocessor_config.json"), False),
    ("MIRAGE", "model.safetensors", hf("MIRAGE", "model.safetensors"), False),
    ("MIRAGE", "config.json", hf("MIRAGE", "config.json"), False),
    ("MIRAGE", "mirage_hf.py", github("j-morano/MIRAGE", "MIRAGE", "hf/mirage_hf.py"), False),
    ("Inception", KERAS, f"https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/{KERAS}", False),
]
assert {(arm, name) for arm, name, _, _ in FILES} == {(arm, name) for arm in ARMS for name in pins["digests"][arm]}


def record(row):
    """Append to the download-cost ledger the protocol asks for (runbook, R4)."""
    path = pathlib.Path(LEDGER)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        if new:
            writer.writerow(["arm", "file", "url", "bytes", "seconds", "utc"])
        writer.writerow(row)


def fetch(arm, name, url, token=None):
    """Download one pinned file. Bytes that fail the digest are deleted, never kept."""
    target = pathlib.Path(ASSETS, arm, name)
    expected = pins["digests"][arm][name]
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256(target) == expected:
        print(f"already verified  {arm}/{name}")
        return
    partial = target.with_name(name + ".part")
    digest, size, start = hashlib.sha256(), 0, time.time()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    with requests.get(url, headers=headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for chunk in response.iter_content(1 << 20):
                stream.write(chunk)
                digest.update(chunk)
                size += len(chunk)
    seconds = round(time.time() - start, 1)
    if digest.hexdigest() != expected:
        partial.unlink()
        raise ValueError(f"{arm}/{name} does not match its pinned SHA256; nothing was kept")
    partial.replace(target)
    record([arm, name, url, size, seconds, datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")])
    print(f"downloaded {size / 1e6:9.1f} MB in {seconds:7.1f} s  {arm}/{name}")

In [ ]:
# Step 5b. Download MIRAGE and the Inception weights (public; no token is read)
for arm, name, url, gated in FILES:
    if arm in G1_ARMS:
        assert not gated, f"{arm}/{name} needs a token; G1 must not read one"
        fetch(arm, name, url)

In [ ]:
# Step 8. Download the external datasets (images only: no encoder, no label used, no score)
BOREALIS = "https://borealisdata.ca/api"
OCTID = {"doi:10.5683/SP/WLW4ZT": "NORMAL", "doi:10.5683/SP/MBMQGD": "MH", "doi:10.5683/SP/FVLZIL": "CSR"}
OCTID_LISTING_SHA256 = "984fde2b5222c44a3c2ed7bf34b5d513a921b7d88d3a12e4518cbb28f52b6f21"  # checked 10 Nov
OCTID_IMAGES = {"NORMAL": 206, "MH": 102, "CSR": 102}
ZENODO = "https://zenodo.org/records/10839556/files"
OCTDL_MD5 = {"OCTDL.zip": "dec9766fc916202c45a333c4f5a09576", "OCTDL_labels.csv": "2fced789038c6ca83283066c2f8ad289"}
OCTDL_IMAGES = {"NO": 332, "VID": 76}
A1_COUNTS = {"OCTDL/MH": 39, "OCTDL/NORMAL": 332, "OCTID/CSC": 102, "OCTID/MH": 102, "OCTID/NORMAL": 206}  # amendment A1
EXTERNAL = pathlib.Path(ROOT, "data/external")
for folder in (META, DOWNLOADS):
    os.makedirs(folder, exist_ok=True)


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_md5(source, url, target, expected):
    """Download one public file. Bytes that fail the published MD5 are deleted, never kept."""
    target = pathlib.Path(target)
    if target.exists() and md5(target) == expected:
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    partial, size, start = target.with_name(target.name + ".part"), 0, time.time()
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for chunk in response.iter_content(1 << 20):
                stream.write(chunk)
                size += len(chunk)
    if md5(partial) != expected:
        partial.unlink()
        raise ValueError(f"{target.name} does not match its published MD5; nothing was kept")
    partial.replace(target)
    record([source, target.name, url, size, round(time.time() - start, 1), datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")])


listing = []
for doi in OCTID:
    response = requests.get(f"{BOREALIS}/datasets/:persistentId/", params={"persistentId": doi}, timeout=120)
    response.raise_for_status()
    version = response.json()["data"]["latestVersion"]
    title = next(field["value"] for field in version["metadataBlocks"]["citation"]["fields"] if field["typeName"] == "title")
    listing += [[doi, title, f["dataFile"]["filename"], f["dataFile"]["filesize"], f["dataFile"]["md5"], f["dataFile"]["id"]] for f in version["files"]]
listing.sort(key=lambda row: (row[0], row[2]))
with open(f"{META}/octid_files.csv", "w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream, lineterminator="\n")
    writer.writerow(["doi", "title", "filename", "bytes", "md5", "file_id"])
    writer.writerows(listing)
assert sha256(f"{META}/octid_files.csv") == OCTID_LISTING_SHA256, "The OCTID release changed since 10 Nov: stop and report."
for doi, _, name, _, digest, file_id in listing:
    if name.lower().endswith(".jpeg"):
        fetch_md5("OCTID", f"{BOREALIS}/access/datafile/{file_id}", EXTERNAL / "OCTID" / OCTID[doi] / name, digest)
found = {folder: len(list((EXTERNAL / "OCTID" / folder).glob("*.jpeg"))) for folder in OCTID_IMAGES}
assert found == OCTID_IMAGES, found

fetch_md5("OCTDL", f"{ZENODO}/OCTDL.zip?download=1", f"{DOWNLOADS}/OCTDL.zip", OCTDL_MD5["OCTDL.zip"])
fetch_md5("OCTDL", f"{ZENODO}/OCTDL_labels.csv?download=1", f"{META}/OCTDL_labels.csv", OCTDL_MD5["OCTDL_labels.csv"])
with zipfile.ZipFile(f"{DOWNLOADS}/OCTDL.zip") as archive:
    for info in archive.infolist():
        parts = info.filename.split("/")
        if not info.is_dir() and len(parts) == 3 and parts[0] == "OCTDL" and parts[1] in OCTDL_IMAGES:
            target = EXTERNAL.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
found = {folder: len(list((EXTERNAL / "OCTDL" / folder).glob("*.jpg"))) for folder in OCTDL_IMAGES}
assert found == OCTDL_IMAGES, found

sh(f"{PY} {ROOT}/fm_external.py candidates --octid-files {META}/octid_files.csv --octdl-labels {META}/OCTDL_labels.csv "
   f"--out {META}/external_candidates.csv")
with open(f"{META}/external_candidates.csv", newline="", encoding="utf-8") as stream:
    counts = {}
    for row in csv.DictReader(stream):
        counts[f"{row['source']}/{row['label']}"] = counts.get(f"{row['source']}/{row['label']}", 0) + 1
assert counts == A1_COUNTS, f"External cohorts differ from amendment A1, stop and report: {counts}"
ELIGIBILITY = ("Amendment A1 (12 November 2026), fixed in fm_external.py: NORMAL queries for the false-rare rate; "
               "OCTID CSR as CSC and OCTID MH; OCTDL disease=VID with subcategory=MH. Queries only.")
EXTERNAL_GATE = {
    "OCTID": dict(status="planned",
                  release="borealisdata.ca/dataverse/OCTID: doi:10.5683/SP/WLW4ZT, SP/MBMQGD, SP/FVLZIL, each v1.0. "
                          "release_sha256 is the SHA256 of the file listing (name, size and MD5 of every file); "
                          "every image was checked against its MD5.",
                  release_sha256=sha256(f"{META}/octid_files.csv"),
                  terms_evidence=f"CC0 1.0 (Borealis dataset pages). {EXTERNAL_DECISION}",
                  eligibility=ELIGIBILITY,
                  label_inventory=f"external_candidates.csv SHA256 {sha256(META + '/external_candidates.csv')}: "
                                  "NORMAL 206, CSC 102, MH 102"),
    "OCTDL": dict(status="planned",
                  release="zenodo.org/records/10839556 (OCTDL.zip and OCTDL_labels.csv, published MD5 verified). "
                          "release_sha256 is the SHA256 of OCTDL.zip.",
                  release_sha256=sha256(f"{DOWNLOADS}/OCTDL.zip"),
                  terms_evidence=f"CC BY 4.0 (Zenodo record). {EXTERNAL_DECISION}",
                  eligibility=ELIGIBILITY,
                  label_inventory=f"OCTDL_labels.csv SHA256 {sha256(META + '/OCTDL_labels.csv')}: NORMAL 332, MH 39"),
}
print(json.dumps(counts, indent=2, sort_keys=True))
print("External images are staged. No encoder has run on them and no external label has been used.")

In [ ]:
# Step 9. Keep Drive and credentials away from the model code (risk record R1)
TOKEN_CACHES = ("/root/.cache/huggingface/token", "/root/.cache/huggingface/stored_tokens", "/root/.huggingface/token")
assert not os.path.ismount("/content/drive"), "Drive is mounted: run drive.flush_and_unmount() first."
cached = [path for path in TOKEN_CACHES if os.path.exists(path)]
assert not cached, f"A Hugging Face login is cached at {cached}. Use a fresh runtime, so the model code cannot read it."
SAFE_ENV = {name: value for name, value in os.environ.items()
            if not any(word in name.upper() for word in ("TOKEN", "SECRET", "KEY", "PASSW", "CREDENTIAL", "HF_", "HUGGING"))}
print("Removed from the worker environment (names only):", sorted(set(os.environ) - set(SAFE_ENV)) or "nothing")
INTERPRETER = {**{arm: PY for arm in ARMS}, "Inception": sys.executable}  # the reference uses Colab's own TensorFlow


def worker(command, log):
    """Run one process with no credentials in its environment, keep its output in `log`, wait until it exits."""
    process = subprocess.Popen(command, env=SAFE_ENV, cwd="/content", stdin=subprocess.DEVNULL,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    os.makedirs(os.path.dirname(log), exist_ok=True)
    with open(log, "w", encoding="utf-8") as stream:
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
    return process.pid, process.wait()


def gpu_users():
    return subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip()

In [ ]:
# Step 8. Pad and extract, one process per model, no credentials in its environment (no labels, no scores)
assert not os.path.ismount("/content/drive"), "Drive is mounted: run drive.flush_and_unmount() first."
for arm in G1_ARMS:
    out = f"{G1}/out/{arm}"
    if not os.path.exists(f"{out}/complete.json"):
        print(f"\n=== {arm} ===", flush=True)
        started = time.time()
        pid, code = worker([INTERPRETER[arm], f"{ROOT}/fm_pad_extract.py", "extract", "--root", ROOT, "--lock", f"{G1}/lock",
                            "--external", f"{G1}/external", "--arm", arm, "--assets", f"{ASSETS}/{arm}", "--out", out],
                           f"{G1}/logs/{arm}.log")
        print(f"{arm}: exit code {code} after {round(time.time() - started, 1)} s", flush=True)
        assert code == 0 and os.path.exists(f"{out}/complete.json"), (
            f"G1 extraction failed for {arm}. The log is {G1}/logs/{arm}.log. Run this step once more, changing nothing; "
            "if it fails again, run Step 9 to save the logs and tell Lam.")

In [ ]:
# Step 9. Save both feature folders and the logs to octfsl/fm_v1/final/<attempt>/g1/, then disconnect Drive
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archives = []
for arm in G1_ARMS:
    if os.path.exists(f"{G1}/out/{arm}/complete.json"):
        archive = f"{G1}/exports/g1-{arm}-{stamp}.zip"
        os.makedirs(os.path.dirname(archive), exist_ok=True)
        sh(f"{PY} {ROOT}/fm_prepare.py export --root {ROOT} --out {G1}/out/{arm} --archive {archive}")
        archives += [archive, archive + ".json"]
logs = f"{G1}/exports/g1-logs-{stamp}.zip"
with zipfile.ZipFile(logs, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(pathlib.Path(G1, "logs").glob("*.log")):
        archive.write(path, path.name)
save_to_drive(f"final/{ATTEMPT}/g1", [*archives, logs])
print(f"All done. Tell Lam: G1 saved in octfsl/fm_v1/final/{ATTEMPT}/g1/ with stamp {stamp}; no score exists yet.")